In [1]:
# Cargar los datos limpios solo para ver los nombres de las columnas
df = catalog.load("df_nhanes_limpio")
variables = df.drop(columns=["Riesgo_Hipertension"]).columns.tolist()

print("Para inventar un paciente, necesitas darle valores a estas 9 variables:")
for var in variables:
    print(f"- {var}")

[06/23/26 16:49:15] INFO     Loading data from df_nhanes_limpio (CSVDataset)...                ]8;id=9173325;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=9173326;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

Para inventar un paciente, necesitas darle valores a estas 9 variables:
- Edad
- Genero
- Peso_kg
- Estatura_cm
- IMC
- Cintura_cm
- Colesterol_Total
- Fumador


In [3]:
import pandas as pd

# 1. Cargamos tu modelo ya entrenado
modelo = catalog.load("modelo_hipertension")

# 2. Inventamos a nuestros pacientes usando los nombres EXACTOS de las columnas
pacientes_inventados = pd.DataFrame([
    {
        # --- PACIENTE 1: "Don Sano" (Joven, buenos indicadores) ---
        "Edad": 28,                 # 28 años
        "Genero": 1,                # 1: Hombre (asumiendo codificación típica)
        "Peso_kg": 72.0,            # 72 kg
        "Estatura_cm": 175.0,       # 1.75 m
        "IMC": 23.5,                # IMC normal
        "Cintura_cm": 82.0,         # Cintura normal
        "Colesterol_Total": 180.0,  # Colesterol óptimo (<200)
        "Fumador": 0                # 0: No fuma
    },
    {
        # --- PACIENTE 2: "Don Riesgo" (Mayor, sobrepeso, fuma) ---
        "Edad": 65,                 # 65 años
        "Genero": 1,                # 1: Hombre
        "Peso_kg": 105.0,           # 105 kg
        "Estatura_cm": 170.0,       # 1.70 m
        "IMC": 36.3,                # Obesidad tipo II
        "Cintura_cm": 112.0,        # Riesgo cardiovascular alto
        "Colesterol_Total": 260.0,  # Colesterol muy alto
        "Fumador": 1                # 1: Sí fuma
    }
])

# 3. Hacemos la predicción asegurándonos de que el orden de las columnas sea correcto
# (A veces XGBoost exige el mismo orden exacto del entrenamiento)
columnas_esperadas = ['Edad', 'Genero', 'Peso_kg', 'Estatura_cm', 'IMC', 'Cintura_cm', 'Colesterol_Total', 'Fumador']
pacientes_inventados = pacientes_inventados[columnas_esperadas]

predicciones = modelo.predict(pacientes_inventados)
probabilidades = modelo.predict_proba(pacientes_inventados)

# 4. Mostramos los resultados
print("🩺 --- RESULTADOS DEL DIAGNÓSTICO --- 🩺\n")
for i, pred in enumerate(predicciones):
    nombre = "Don Sano" if i == 0 else "Don Riesgo"
    estado = "EN RIESGO (1)" if pred == 1 else "SANO (0)"
    prob_riesgo = probabilidades[i][1] * 100
    print(f"Paciente {nombre}: El modelo indica que está {estado}")
    print(f"   ↳ Seguridad del modelo: {prob_riesgo:.1f}% de probabilidad de hipertensión.\n")

[06/23/26 16:50:28] INFO     Loading data from modelo_hipertension (PickleDataset)...          ]8;id=9173337;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=9173338;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

🩺 --- RESULTADOS DEL DIAGNÓSTICO --- 🩺

Paciente Don Sano: El modelo indica que está SANO (0)
   ↳ Seguridad del modelo: 15.9% de probabilidad de hipertensión.

Paciente Don Riesgo: El modelo indica que está EN RIESGO (1)
   ↳ Seguridad del modelo: 82.6% de probabilidad de hipertensión.



In [4]:
import pandas as pd

# 1. Asegurar que el modelo esté cargado
modelo = catalog.load("modelo_hipertension")

# 2. Creamos los 5 pacientes con perfiles de riesgo incrementales (1 al 5)
pacientes_espectro = pd.DataFrame([
    {
        # --- RIESGO NIVEL 1: El paciente ideal (Joven, atleta, sin vicios) ---
        "Edad": 22,                 
        "Genero": 0,                # Mujer
        "Peso_kg": 58.0,            
        "Estatura_cm": 165.0,       
        "IMC": 21.3,                # Rango saludable
        "Cintura_cm": 74.0,         # Muy saludable
        "Colesterol_Total": 150.0,  # Excelente
        "Fumador": 0                
    },
    {
        # --- RIESGO NIVEL 2: Riesgo Bajo-Moderado (Adulto joven, sobrepeso ligero) ---
        "Edad": 35,                 
        "Genero": 1,                # Hombre
        "Peso_kg": 82.0,            
        "Estatura_cm": 178.0,       
        "IMC": 25.9,                # Sobrepeso leve
        "Cintura_cm": 89.0,         
        "Colesterol_Total": 195.0,  # Cerca del límite (<200)
        "Fumador": 0                
    },
    {
        # --- RIESGO NIVEL 3: Riesgo Medio (Edad madura, fumador, colesterol al límite) ---
        "Edad": 48,                 
        "Genero": 0,                # Mujer
        "Peso_kg": 76.0,            
        "Estatura_cm": 160.0,       
        "IMC": 29.7,                # Al borde de la obesidad
        "Cintura_cm": 94.0,         # Riesgo abdominal moderado para mujer
        "Colesterol_Total": 215.0,  # Colesterol elevado
        "Fumador": 1                # Fumadora activa
    },
    {
        # --- RIESGO NIVEL 4: Riesgo Alto (Adulto mayor, obesidad tipo 1, colesterol alto) ---
        "Edad": 58,                 
        "Genero": 1,                # Hombre
        "Peso_kg": 96.0,            
        "Estatura_cm": 172.0,       
        "IMC": 32.4,                # Obesidad Clase I
        "Cintura_cm": 106.0,        # Riesgo cardiovascular elevado
        "Colesterol_Total": 240.0,  # Alto riesgo por colesterol
        "Fumador": 0                
    },
    {
        # --- RIESGO NIVEL 5: Riesgo Crítico (Adulto mayor, obesidad severa, fumador, pésimos indicadores) ---
        "Edad": 72,                 
        "Genero": 1,                # Hombre
        "Peso_kg": 115.0,           
        "Estatura_cm": 168.0,       
        "IMC": 40.7,                # Obesidad Mórbida (Clase III)
        "Cintura_cm": 124.0,        # Cintura en rango crítico
        "Colesterol_Total": 285.0,  # Colesterol extremadamente alto
        "Fumador": 1                # Fumador crónico
    }
])

# 3. Ordenar las columnas exactamente como las espera el modelo XGBoost
columnas_esperadas = ['Edad', 'Genero', 'Peso_kg', 'Estatura_cm', 'IMC', 'Cintura_cm', 'Colesterol_Total', 'Fumador']
pacientes_espectro = pacientes_espectro[columnas_esperadas]

# 4. Predicciones
predicciones = modelo.predict(pacientes_espectro)
probabilidades = modelo.predict_proba(pacientes_espectro)

# 5. Desplegar los resultados detallados
print("🧪 --- SIMULACIÓN DEL ESPECTRO DE RIESGO DE HIPERTENSIÓN --- 🧪\n")
for i in range(5):
    pred = predicciones[i]
    prob_riesgo = probabilidades[i][1] * 100
    diagnostico = "🚨 EN RIESGO (1)" if pred == 1 else "✅ SANO (0)"
    
    print(f"Nivel de Riesgo Teórico {i+1}:")
    print(f"   ↳ Dictamen del modelo: {diagnostico}")
    print(f"   ↳ Probabilidad calculada por XGBoost: {prob_riesgo:.1f}%")
    print("-" * 50)

[06/23/26 16:52:33] INFO     Loading data from modelo_hipertension (PickleDataset)...          ]8;id=9173343;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=9173344;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

🧪 --- SIMULACIÓN DEL ESPECTRO DE RIESGO DE HIPERTENSIÓN --- 🧪

Nivel de Riesgo Teórico 1:
   ↳ Dictamen del modelo: 🚨 EN RIESGO (1)
   ↳ Probabilidad calculada por XGBoost: 51.4%
--------------------------------------------------
Nivel de Riesgo Teórico 2:
   ↳ Dictamen del modelo: 🚨 EN RIESGO (1)
   ↳ Probabilidad calculada por XGBoost: 55.2%
--------------------------------------------------
Nivel de Riesgo Teórico 3:
   ↳ Dictamen del modelo: 🚨 EN RIESGO (1)
   ↳ Probabilidad calculada por XGBoost: 54.7%
--------------------------------------------------
Nivel de Riesgo Teórico 4:
   ↳ Dictamen del modelo: 🚨 EN RIESGO (1)
   ↳ Probabilidad calculada por XGBoost: 83.4%
--------------------------------------------------
Nivel de Riesgo Teórico 5:
   ↳ Dictamen del modelo: 🚨 EN RIESGO (1)
   ↳ Probabilidad calculada por XGBoost: 79.9%
--------------------------------------------------


In [5]:
# 1. Definimos nuestro nuevo nivel de exigencia (60%)
umbral_personalizado = 0.60

print(f"🧪 --- RESULTADOS CON UMBRAL DEL {umbral_personalizado*100}% --- 🧪\n")

# 2. Re-evaluamos a los 5 pacientes usando el nuevo umbral
for i in range(5):
    # Extraemos la probabilidad de la clase 1 (Riesgo) que calculó XGBoost antes
    prob_riesgo = probabilidades[i][1] 
    
    # Aplicamos nuestra nueva regla lógica
    if prob_riesgo >= umbral_personalizado:
        diagnostico = "🚨 EN RIESGO (1)"
    else:
        diagnostico = "✅ SANO (0)"
        
    print(f"Nivel de Riesgo Teórico {i+1}:")
    print(f"   ↳ Probabilidad original: {prob_riesgo*100:.1f}%")
    print(f"   ↳ Nuevo Diagnóstico: {diagnostico}")
    print("-" * 50)

🧪 --- RESULTADOS CON UMBRAL DEL 60.0% --- 🧪

Nivel de Riesgo Teórico 1:
   ↳ Probabilidad original: 51.4%
   ↳ Nuevo Diagnóstico: ✅ SANO (0)
--------------------------------------------------
Nivel de Riesgo Teórico 2:
   ↳ Probabilidad original: 55.2%
   ↳ Nuevo Diagnóstico: ✅ SANO (0)
--------------------------------------------------
Nivel de Riesgo Teórico 3:
   ↳ Probabilidad original: 54.7%
   ↳ Nuevo Diagnóstico: ✅ SANO (0)
--------------------------------------------------
Nivel de Riesgo Teórico 4:
   ↳ Probabilidad original: 83.4%
   ↳ Nuevo Diagnóstico: 🚨 EN RIESGO (1)
--------------------------------------------------
Nivel de Riesgo Teórico 5:
   ↳ Probabilidad original: 79.9%
   ↳ Nuevo Diagnóstico: 🚨 EN RIESGO (1)
--------------------------------------------------


In [6]:
import pandas as pd

# 1. Recargamos el modelo (Kedro cargará la versión nueva con umbral 0.60 que acabas de guardar)
modelo = catalog.load("modelo_hipertension")

# 2. Volvemos a armar los mismos 5 pacientes
pacientes_espectro = pd.DataFrame([
    {"Edad": 22, "Genero": 0, "Peso_kg": 58.0, "Estatura_cm": 165.0, "IMC": 21.3, "Cintura_cm": 74.0, "Colesterol_Total": 150.0, "Fumador": 0}, # Nivel 1
    {"Edad": 35, "Genero": 1, "Peso_kg": 82.0, "Estatura_cm": 178.0, "IMC": 25.9, "Cintura_cm": 89.0, "Colesterol_Total": 195.0, "Fumador": 0}, # Nivel 2
    {"Edad": 48, "Genero": 0, "Peso_kg": 76.0, "Estatura_cm": 160.0, "IMC": 29.7, "Cintura_cm": 94.0, "Colesterol_Total": 215.0, "Fumador": 1}, # Nivel 3
    {"Edad": 58, "Genero": 1, "Peso_kg": 96.0, "Estatura_cm": 172.0, "IMC": 32.4, "Cintura_cm": 106.0, "Colesterol_Total": 240.0, "Fumador": 0}, # Nivel 4
    {"Edad": 72, "Genero": 1, "Peso_kg": 115.0, "Estatura_cm": 168.0, "IMC": 40.7, "Cintura_cm": 124.0, "Colesterol_Total": 285.0, "Fumador": 1}  # Nivel 5
])

columnas_esperadas = ['Edad', 'Genero', 'Peso_kg', 'Estatura_cm', 'IMC', 'Cintura_cm', 'Colesterol_Total', 'Fumador']
pacientes_espectro = pacientes_espectro[columnas_esperadas]

# 3. Predicciones directas del nuevo modelo
predicciones = modelo.predict(pacientes_espectro)
probabilidades = modelo.predict_proba(pacientes_espectro)

print("🩺 --- EVALUACIÓN DE PACIENTES CON EL MODELO OFICIAL --- 🩺\n")
for i in range(5):
    pred = predicciones[i]
    prob_riesgo = probabilidades[i][1] * 100
    
    # Lógica basada en cómo reaccionará el modelo entrenado con scale_pos_weight
    diagnostico = "🚨 EN RIESGO (1)" if prob_riesgo >= 60.0 else "✅ SANO (0)"
    
    print(f"Paciente Nivel {i+1}: Predict Oficial -> {diagnostico} (Probabilidad: {prob_riesgo:.1f}%)")

[06/23/26 17:16:10] INFO     Loading data from modelo_hipertension (PickleDataset)...          ]8;id=9173349;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=9173350;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

🩺 --- EVALUACIÓN DE PACIENTES CON EL MODELO OFICIAL --- 🩺

Paciente Nivel 1: Predict Oficial -> ✅ SANO (0) (Probabilidad: 51.4%)
Paciente Nivel 2: Predict Oficial -> ✅ SANO (0) (Probabilidad: 55.2%)
Paciente Nivel 3: Predict Oficial -> ✅ SANO (0) (Probabilidad: 54.7%)
Paciente Nivel 4: Predict Oficial -> 🚨 EN RIESGO (1) (Probabilidad: 83.4%)
Paciente Nivel 5: Predict Oficial -> 🚨 EN RIESGO (1) (Probabilidad: 79.9%)
